# Exploring the GIS test data (permits, service requests, road segments)

There is no data dictionary, so before setting up the ArcGIS Pro project I checked the three layers for the decisions the geoprocessing depends on:

1. **Load:** what's in each layer, and are the key columns complete?
2. **Permits (layer 23):** which columns mean renewal, issued, excavation and paving, and how many permits pass?
3. **Service requests (Cityworks):** does the provided SQL match the brief, and how many requests pass?
4. **Road segments (layer 162):** which road types count as streets, service lanes and ramps?
5. **Snapping:** how far are the permit points from the road?

Each section ends with the decision it led to.

*This notebook uses the Python test's environment (`python_test/.venv`, kernel "CS12 Python Test (3.11)"), since it needs the same two packages, `requests` and `pandas`. No `arcpy`, so it runs outside ArcGIS Pro.*

In [1]:
import pandas as pd
import requests

pd.set_option("display.width", 160)

PERMITS_23 = "https://maps2.dcgis.dc.gov/dcgis/rest/services/DCGIS_APPS/DDOT_TOPS/MapServer/23"
REQUESTS_CW = "https://maps2.dcgis.dc.gov/dcgis/rest/services/DDOT/Cityworks/MapServer/1"
SUBBLOCKS_162 = "https://maps2.dcgis.dc.gov/dcgis/rest/services/DCGIS_DATA/Transportation_WebMercator/MapServer/162"

## 1. Load the data
 The services hand out at most 2,000 rows per request, so it asks page by page.

- **Permits:** the whole layer (about 55,000 rows), so nothing is hidden before I decide how to filter.
- **Service requests:** Cityworks holds about 1.5 million requests going back years. The brief only wants requests created on or after 1 October 2025, so only those are downloaded (about 75,000). Every other filter is decided from the data.
- **Road segments:** the whole layer (about 35,000), without the line shapes for now.

In [2]:
def download(layer_url, where="1=1"):
    # The layer's own description: column names, types and code lists
    layer_info = requests.get(layer_url, params={"f": "json"}, timeout=60).json()

    # Ask for the rows page by page until the service says there are no more
    rows, offset = [], 0
    while True:
        params = {
            "where": where,
            "outFields": "*",
            "returnGeometry": "false",    # the points have their coordinates as columns
            "orderByFields": "OBJECTID",  # same order every time, so pages don't overlap
            "resultOffset": offset,
            "resultRecordCount": 2000,
            "f": "json",
        }
        page = requests.get(f"{layer_url}/query", params=params, timeout=120).json()
        if "error" in page:               # a bad query still comes back as "OK"
            raise RuntimeError(page["error"])
        rows += [feature["attributes"] for feature in page["features"]]
        if not page.get("exceededTransferLimit"):
            break
        offset += len(page["features"])
    table = pd.DataFrame(rows)

    # Dates arrive as milliseconds since 1970 in UTC, so convert them to DC time.
    # Coded columns get an extra "_label" column with the readable value.
    for field in layer_info["fields"]:
        name = field["name"]
        if name not in table:
            continue
        if field["type"] == "esriFieldTypeDate":
            table[name] = pd.to_datetime(table[name], unit="ms", utc=True).dt.tz_convert("America/New_York")
        domain = field.get("domain") or {}
        if domain.get("type") == "codedValue":
            labels = {str(c["code"]): c["name"] for c in domain["codedValues"]}
            table[name + "_label"] = table[name].astype("string").str.removesuffix(".0").map(labels)
    return table

In [3]:
permits = download(PERMITS_23)
service_requests = download(REQUESTS_CW, where="INITIATEDDATE >= timestamp '2025-10-01 00:00:00'")
subblocks = download(SUBBLOCKS_162)

for name, table in [("Permits (layer 23)", permits), ("Service requests since 1 Oct 2025", service_requests),
                    ("Road segments (layer 162)", subblocks)]:
    print(f"{name}: {len(table):,} rows, {table.shape[1]} columns")

Permits (layer 23): 54,625 rows, 102 columns
Service requests since 1 Oct 2025: 74,944 rows, 57 columns
Road segments (layer 162): 34,964 rows, 113 columns


In [4]:
# How complete are the columns this task depends on? (share of rows filled)
permit_columns = ["TrackingNumber", "PermitNumber", "StatusDescription", "IssueDate", "ExpirationDate",
                  "Renewal", "RenewalDescription", "IsExcavation", "IsPaving", "PermitTypeDescription",
                  "WorkLocationFullAddress", "XCoord", "YCoord", "OnSegX", "OnSegY", "StreetSegID", "RoadwaySegID"]
request_columns = ["REQUESTID", "DESCRIPTION", "STATUS", "INITIATEDDATE", "INSPECTIONDATE", "INSPECTIONCOMPLETE",
                   "CLOSEDDATE", "ADDRESS", "XCOORD", "YCOORD", "ONSEGX", "ONSEGY"]
subblock_columns = ["SUBBLOCKKEY", "ROUTENAME", "ROADTYPE"]

pd.concat([
    permits[permit_columns].notna().mean().round(3).rename("share filled").to_frame().assign(layer="permits"),
    service_requests[request_columns].notna().mean().round(3).rename("share filled").to_frame().assign(layer="service requests"),
    subblocks[subblock_columns].notna().mean().round(3).rename("share filled").to_frame().assign(layer="road segments"),
])

,share filled,layer
TrackingNumber,1.000,permits
PermitNumber,0.659,permits
StatusDescription,1.000,permits
IssueDate,0.659,permits
ExpirationDate,1.000,permits
Renewal,1.000,permits
RenewalDescription,1.000,permits
IsExcavation,1.000,permits
IsPaving,1.000,permits
PermitTypeDescription,0.866,permits


**What this shows**

- All three layers download cleanly: 54,625 permits, 74,944 service requests since 1 October 2025, and 34,964 road segments.
- The columns the filters need are always filled: `ExpirationDate`, `Renewal`, `IsExcavation` and `IsPaving` for permits; `DESCRIPTION`, `STATUS`, `INITIATEDDATE` and `INSPECTIONCOMPLETE` for requests; `ROADTYPE` for road segments.
- `IssueDate` and `PermitNumber` are filled on 66% of permits. As in layer 24, they probably only exist once a permit is issued, which section 2 checks.
- Permits can't be joined to road segments by an ID: `RoadwaySegID` is always empty and `StreetSegID` is filled on only 59%, and neither is the road segment key (`SUBBLOCKKEY`).

**Decision:** permits and requests are linked to road segments **by location** (snapping and a spatial join in ArcGIS Pro), not by a shared ID.

## 2. Permits: which rows count

The brief's three permit filters: expiration on or after 1 October 2025, only *renewal* permits that have been *issued*, and only excavation and paving permits. For each one, I check how it shows up in the data.

In [5]:
# Renewal: what do the codes mean?
pd.crosstab(permits["Renewal"], permits["RenewalDescription"])

RenewalDescription,New,Renewal
Renewal,,
1,27259,0
2,0,27366


`Renewal` is a code, and **2 means Renewal** (1 means New). So the filter is `Renewal = 2`, not `Renewal = 1`.

In [6]:
# Issued: which statuses have an issue date?
pd.crosstab(permits["StatusDescription"], permits["IssueDate"].notna().rename("has issue date"), margins=True)

has issue date,False,True,All
StatusDescription,,,
Approved (Pending Payment),11804,0,11804
Assigned,1386,0,1386
Cancel/Withdrawn,1138,0,1138
Denied,73,0,73
Issued,0,21452,21452
Not Paid,970,0,970
Pending Assignment,44,0,44
Permit Expired,23,14560,14583
Reopened,4,0,4


An issued permit keeps its issue date when its status moves on. Every "Issued" permit has an issue date, but so do 14,560 "Permit Expired" permits (issued, then expired) and a few "Suspended" and "Revoked" ones. The 23 "Permit Expired" rows without an issue date expired before they were ever issued.

**Decision:** "has been issued" = **has an issue date** (`IssueDate IS NOT NULL`), whatever the current status. Filtering on `StatusDescription = 'Issued'` alone would drop every issued permit that has since expired.

In [7]:
# Excavation and paving: how do the two flags combine?
print(pd.crosstab(permits["IsExcavation"], permits["IsPaving"]))

# What are the permits that are neither?
neither = permits[(permits["IsExcavation"] == 0) & (permits["IsPaving"] == 0)]
neither["TypeDetailNames"].str.split(";").str[0].value_counts().head(8)

IsPaving          0    1
IsExcavation            
0              1328  943
1             52222  132


TypeDetailNames
Over Head Work    676
Fixture           438
Projections       112
Landscaping        70
Mega Projects      30
Annual Permits      1
PS Rental           1
Name: count, dtype: int64

Most permits are excavation (96%), a few are paving, and only a handful are both. The permits that are neither are other work types such as overhead work and fixtures.

**Decision:** "excavation and paving permits" = **either flag** (`IsExcavation = 1 OR IsPaving = 1`). Requiring both would leave almost nothing.

In [8]:
# Expiration: what range of dates does the layer hold?
print("Earliest expiration:", permits["ExpirationDate"].min())
print("Latest expiration:  ", permits["ExpirationDate"].max())

start = pd.Timestamp("2025-10-01", tz="America/New_York")
print(f"Expiring on/after 1 Oct 2025: {(permits['ExpirationDate'] >= start).mean():.0%}")

Earliest expiration: 2026-08-31 04:00:00-04:00
Latest expiration:   2039-05-05 05:00:00-04:00
Expiring on/after 1 Oct 2025: 100%


Every permit in the layer expires after 1 October 2025. As with the occupancy permits in the Python test, the layer seems to drop permits once they have expired for a while. So the date filter is kept as the brief asks, but removes nothing in this pull.

In [9]:
# All filters together: how many rows and permits are left after each one?
steps = {
    "All rows": pd.Series(True, index=permits.index),
    "Expires on/after 1 Oct 2025": permits["ExpirationDate"] >= start,
    "Renewal (Renewal = 2)": permits["Renewal"] == 2,
    "Issued (has an issue date)": permits["IssueDate"].notna(),
    "Excavation or paving": (permits["IsExcavation"] == 1) | (permits["IsPaving"] == 1),
}
keep = pd.Series(True, index=permits.index)
counts = []
for step, rule in steps.items():
    keep &= rule
    counts.append({"filter": step, "rows": keep.sum(), "permits": permits.loc[keep, "TrackingNumber"].nunique()})
print(pd.DataFrame(counts).to_string(index=False))

filtered_permits = permits[keep]
print()
print(filtered_permits["StatusDescription"].value_counts())
print()
print("Rows per permit:", filtered_permits.groupby("TrackingNumber").size().describe().round(1).to_dict())

                     filter  rows  permits
                   All rows 54625    10191
Expires on/after 1 Oct 2025 54625    10191
      Renewal (Renewal = 2) 27366     4547
 Issued (has an issue date) 19167     3814
       Excavation or paving 18772     3552



StatusDescription
Permit Expired    10837
Issued             7934
Suspended             1
Name: count, dtype: int64

Rows per permit: {'count': 3552.0, 'mean': 5.3, 'std': 10.3, 'min': 1.0, '25%': 1.0, '50%': 1.0, '75%': 4.0, 'max': 176.0}


**What this shows**

- After all filters: about 18,800 rows from about 3,550 permits. Note that 58% of them now have the status "Permit Expired", which confirms the issue-date rule matters.
- **One permit can have many rows**, one per location (median 1, up to 176). This matters for "count of permits on each subblock": a permit with 10 locations on the same block should probably count once, not 10 times. Section 5 looks at this once the points are matched to blocks.

**Decision: the permit definition query for ArcGIS Pro**

```sql
ExpirationDate >= timestamp '2025-10-01 00:00:00'
AND Renewal = 2
AND IssueDate IS NOT NULL
AND (IsExcavation = 1 OR IsPaving = 1)
```

## 3. Service requests: does the provided SQL match the brief?

The brief describes the requests in words (open, pending, or completed inspection; roadway repair, pothole or paving requests) and also gives a SQL definition query:

```sql
STATUS NOT IN ('CLOSED', 'CANCEL', 'COMPLETE')
And DESCRIPTION IN ('ROADWAY REPAIR', 'POTHOLES', 'STREET REPAVING', 'STREET REPAIR', 'STREET PAVING SCHEDU')
And INITIATEDDATE >= timestamp '2025-10-01 00:00:00'
```

The date part was applied when downloading. Here I check the other two parts against the words.

In [10]:
# Statuses: which ones does the SQL keep, and which does the brief name?
statuses = service_requests["STATUS"].value_counts().to_frame("requests")
statuses["kept by SQL"] = ~statuses.index.isin(["CLOSED", "CANCEL", "COMPLETE"])
statuses

,requests,kept by SQL
STATUS,,
CLOSED,65761,False
PENDING,4775,True
INSPCOMP,2344,True
OPEN,1718,True
INSPECTION COMPLETED,208,True
COMPLETE,109,False
GEO ISSUE,29,True


The SQL lists the statuses to **exclude**, so everything else stays in. That covers the brief's three: OPEN, PENDING, and inspection completed, which appears under two spellings, `INSPCOMP` and `INSPECTION COMPLETED`. It also keeps **GEO ISSUE**, which the brief doesn't mention (probably a location problem flagged on the request).

In [11]:
# Request types: which of the SQL's descriptions actually appear?
sql_types = ["ROADWAY REPAIR", "POTHOLES", "STREET REPAVING", "STREET REPAIR", "STREET PAVING SCHEDU"]
print(service_requests["DESCRIPTION"].value_counts().reindex(sql_types, fill_value=0))

# Any other road or paving types the SQL might be missing?
descriptions = service_requests["DESCRIPTION"].value_counts()
descriptions[descriptions.index.str.contains("ROAD|STREET|PAV|POTH|ASPHALT|SINKHOLE")]

DESCRIPTION
ROADWAY REPAIR          1454
POTHOLES                5751
STREET REPAVING            0
STREET REPAIR              0
STREET PAVING SCHEDU       0
Name: count, dtype: int64


DESCRIPTION
ROADWAY SIGN      8779
POTHOLES          5751
ROADWAY REPAIR    1454
Name: count, dtype: int64

Since 1 October 2025 only two of the five listed types occur: POTHOLES and ROADWAY REPAIR. The three paving types have no requests in this period. That's harmless, and they stay in the query in case they appear later. No other road repair or paving type is missing (ROADWAY SIGN is about signs, not the road surface).

In [12]:
# The provided SQL, applied to the downloaded requests
road_requests = service_requests[
    ~service_requests["STATUS"].isin(["CLOSED", "CANCEL", "COMPLETE"])
    & service_requests["DESCRIPTION"].isin(sql_types)
]
print(f"Requests passing the provided SQL: {len(road_requests)}")
print(pd.crosstab(road_requests["DESCRIPTION"], road_requests["STATUS"], margins=True))
print()
print("One row per request:", road_requests["REQUESTID"].is_unique)

Requests passing the provided SQL: 692
STATUS          GEO ISSUE  INSPCOMP  OPEN  PENDING  All
DESCRIPTION                                            
POTHOLES                0         0     0       53   53
ROADWAY REPAIR          1       298   205      135  639
All                     1       298   205      188  692

One row per request: True


**What this shows**

- The provided SQL leaves **692 requests**: 639 roadway repair and 53 potholes. Potholes are common (5,751 since October) but most are closed quickly, so few are still open.
- Only one passing request has the status GEO ISSUE. Following the brief's words strictly would drop it (691).
- Each request is one row, so counting rows per block counts requests (unlike permits, which have a row per location).

**Decision:** use the provided SQL **as given** in ArcGIS Pro. It's DDOT's own definition, and it matches the brief's words except for the one GEO ISSUE request, which I note in the write-up rather than change.

## 4. Road segments: streets, service lanes and ramps

The brief keeps only streets, service lanes and ramps. `ROADTYPE` is a coded column, so the download added a readable `ROADTYPE_label`.

In [13]:
# Road types: code, name and number of segments
subblocks.groupby(["ROADTYPE", "ROADTYPE_label"]).size().to_frame("segments")

,,segments
ROADTYPE,ROADTYPE_label,
1,Street,22393
2,Service Road,121
3,Ramp,590
4,Alley,9076
5,Driveway,2421
6,Trail,345
7,Walkway,18


The code list has no "service lane", but it has **Service Road** (code 2), which is the same thing. Streets are code 1 and ramps code 3. Alleys, driveways, trails, walkways and streetcar tracks are left out.

In [14]:
# ROADTYPE is stored as text ("1", not 1), so the codes need quotes
kept_types = ["1", "2", "3"]  # Street, Service Road, Ramp
road_segments = subblocks[subblocks["ROADTYPE"].isin(kept_types)]
print(f"Road segments kept: {len(road_segments):,} of {len(subblocks):,}")
print("One row per segment:", road_segments["SUBBLOCKKEY"].is_unique)

Road segments kept: 23,104 of 34,964


One row per segment: True


**Decision: the road segment definition query for ArcGIS Pro**

```sql
ROADTYPE IN ('1', '2', '3')
```

The quotes matter: `ROADTYPE` is a text column holding number codes, so `ROADTYPE IN (1, 2, 3)` matches nothing.

## 5. Snapping: how far are the permits from the road?

The Deputy Chief has noticed that some permits aren't snapped to the roadway. The layer has two positions per permit: `XCoord`/`YCoord` (the work location, often a house) and `OnSegX`/`OnSegY` (DDOT's own position on the street). The points on the map are drawn at `OnSegX`/`OnSegY`, which I checked by comparing the layer's geometry with both pairs of columns.

In [15]:
# How often did DDOT actually move the point onto the street?
moved = (filtered_permits["XCoord"] - filtered_permits["OnSegX"]).abs().add(
         (filtered_permits["YCoord"] - filtered_permits["OnSegY"]).abs()) > 0.01
print(f"Map point differs from the work location: {moved.mean():.0%}")
print(f"Map point is just the work location again: {(~moved).mean():.0%}")

Map point differs from the work location: 90%
Map point is just the work location again: 10%


To measure the distance from each map point to the nearest road centreline, I use DDOT's road lookup service (`getPointOnSubBlock`, the same service as in the Python test). It returns the road segments near a point, with their distance. Checking all 14,500 distinct points would take about an hour, so I check a fixed random sample of 400.

In [16]:
from concurrent.futures import ThreadPoolExecutor

LRS = ("https://rh.dcgis.dc.gov/dcgis/rest/services/DDOT/RoadwayBlocks/MapServer/"
       "exts/DdotLrsExtensions/getPointOnSubBlock")
NOT_ROADS = ["Alley", "Driveway", "Walkway", "Trail", "Unknown"]

def nearest_road(x, y):
    # Nearest street, service road or ramp within 100 m of the point, and its distance
    params = {"x": x, "y": y, "inSR": 26985, "outSR": 26985, "searchRadius": 100, "f": "json"}
    found = sorted(requests.get(LRS, params=params, timeout=60).json().get("pointOnSubBlocks", []),
                   key=lambda s: s["offsetInMeters"])
    roads = [s for s in found if s["routeType"] not in NOT_ROADS]
    # Also note the nearest segment of any type (alleys included), to explain off-road points
    answer = {"nearest_any_type": found[0]["routeType"] if found else None,
              "road": None, "distance_m": None}
    if roads:
        answer["road"] = " ".join(roads[0]["routeName"].split())
        answer["distance_m"] = roads[0]["offsetInMeters"]
    return answer

points = filtered_permits.assign(moved=moved).drop_duplicates(["OnSegX", "OnSegY"]).sample(400, random_state=1)
with ThreadPoolExecutor(max_workers=5) as pool:
    answers = list(pool.map(nearest_road, points["OnSegX"], points["OnSegY"]))
points = points.reset_index(drop=True).join(pd.DataFrame(answers))

points["distance_m"].describe(percentiles=[0.5, 0.65, 0.75, 0.9, 0.95, 0.99]).round(1)

count    400.0
mean      10.0
std       15.7
min        0.0
50%        0.0
65%        0.0
75%       21.6
90%       35.9
95%       41.7
99%       51.1
max       63.8
Name: distance_m, dtype: float64

In [17]:
# Is a point on the road (within 0.5 m) or off it? And does that depend on whether DDOT moved it?
points["on_road"] = points["distance_m"] <= 0.5
print(pd.crosstab(points["moved"].map({True: "moved by DDOT", False: "not moved"}),
                  points["on_road"].map({True: "on the road", False: "off the road"}), margins=True))
print()
print("Off-road points, distance to the nearest road (m):")
print(points.loc[~points["on_road"], "distance_m"].describe(percentiles=[0.5, 0.9, 0.95]).round(1))

on_road        off the road  on the road  All
moved                                        
moved by DDOT           127          247  374
not moved                 0           26   26
All                     127          273  400

Off-road points, distance to the nearest road (m):
count    127.0
mean      31.4
std       10.0
min       14.0
50%       30.0
90%       44.7
95%       48.9
max       63.8
Name: distance_m, dtype: float64


In [18]:
# The off-road points: what kind of line are they sitting on?
points.loc[~points["on_road"], "nearest_any_type"].value_counts()

nearest_any_type
Alley       124
Driveway      3
Name: count, dtype: int64

In [19]:
# Snapping to the nearest road: is it the street in the permit's address?
def street_in_address(address):
    # "929 DIVISION AVENUE NE" -> "DIVISION AVE NE", to compare with road names
    street = re.sub(r"^\d+(\s*-\s*\d+)?\s+(BLOCK OF\s+)?", "", str(address).upper())
    short = {"STREET": "ST", "AVENUE": "AVE", "PLACE": "PL", "ROAD": "RD", "DRIVE": "DR", "COURT": "CT",
             "TERRACE": "TER", "BOULEVARD": "BLVD", "LANE": "LN", "CIRCLE": "CIR", "PARKWAY": "PKWY"}
    return " ".join(short.get(word, word) for word in street.split())

import re
off_road = points[~points["on_road"] & points["road"].notna()].copy()
off_road["same_street"] = off_road["road"] == off_road["WorkLocationFullAddress"].map(street_in_address)
off_road["distance_band"] = pd.cut(off_road["distance_m"], [0, 10, 20, 30, 40, 60, 100])
off_road.groupby("distance_band", observed=True)["same_street"].agg(["size", "mean"]).round(2)

,size,mean
distance_band,,
"(10, 20]",16,0.94
"(20, 30]",47,0.66
"(30, 40]",39,0.51
"(40, 60]",24,0.67
"(60, 100]",1,0.00


**What this shows**

- DDOT moved 90% of the map points away from the work location, onto a line. Two thirds of the sample sits exactly on a street centreline.
- The other third (127 of 400) is **off the street, but sitting exactly on an alley or a driveway**. DDOT did snap these permits, just onto the wrong kind of line: the alley behind the house instead of the street in front. This is the "not snapped to the roadway" problem.
- Those points are 14 to 64 m from the nearest street (median 30 m).
- The nearest street is the address street on 94% of points within 20 m, but only on 51–67% of points further away. From an alley, a side street is often closer than the front street.

**Decision: snap in ArcGIS Pro to the filtered road segments only** (streets, service roads and ramps, `ROADTYPE IN ('1', '2', '3')`), so the alleys can't attract the points again. Snap distance **70 m**: it covers every off-road point in the sample (furthest 64 m), and points already on a street don't move. The snap distance is recorded per point, and points moved more than 20 m are flagged for checking, since that's where the wrong-street risk is highest.

In [20]:
# Count of permits per block: a permit often lists several houses on the same block
example = filtered_permits[filtered_permits["WorkLocationFullAddress"].str.contains("DIVISION AVENUE NE", na=False)]
print(example.groupby("TrackingNumber")["WorkLocationFullAddress"].apply(list).head(3).to_string())
print()
same_point = filtered_permits.duplicated(["TrackingNumber", "OnSegX", "OnSegY"]).sum()
print(f"Rows repeating a point already listed in the same permit: {same_point}")
print(f"Rows: {len(filtered_permits):,}  Permits: {filtered_permits['TrackingNumber'].nunique():,}")

TrackingNumber
476767    [929 DIVISION AVENUE NE, 931 DIVISION AVENUE N...
500858                             [801 DIVISION AVENUE NE]
501053    [929 DIVISION AVENUE NE, 931 DIVISION AVENUE N...

Rows repeating a point already listed in the same permit: 128
Rows: 18,772  Permits: 3,552


One permit often covers several houses on the same block (e.g. 929, 931, 933… Division Ave NE under one permit). Counting rows per block would count that one permit five or six times.

**Decision:** "count of permits on each subblock" = **number of distinct permits** (`TrackingNumber`) on the block, not the number of permit rows. In ArcGIS Pro this means joining permits to blocks first, then counting unique tracking numbers per block. The row count can be kept as a second column (`PermitLocations`).